# 05 — SetFit-only Sentiment, Churn Risk & RAG JSON Export

This version uses **SetFit as the only production intent model**.

Reason for the choice:
- DistilBERT performs very well on the internal test split, but generalizes poorly on external datasets.
- SetFit has lower internal metrics, but better external performance on CLINC150 and Kaggle Ecommerce Intent.
- For a RAG routing system, external generalization and semantic robustness are more important than internal benchmark score.

Pipeline:
1. Load the test split and SetFit model
2. Run SetFit intent inference with confidence and top-k predictions
3. Evaluate SetFit on the internal test split
4. Add sentiment with RoBERTa
5. Derive churn risk from intent + confidence
6. Export a JSON file for RAG

### Input files expected
```text
data/splits/test.csv
models/setfit_intent/best_model/
models/setfit_intent/best_model/label_mappings.json
```

### Output files produced
```text
reports/rag_export/rag_output.json
reports/rag_export/setfit_internal_metrics.json
reports/rag_export/label_taxonomy.json
```

## 1. Imports and configuration

In [1]:
#!pip install -q setfit sentence-transformers transformers datasets evaluate

In [2]:
import torch, gc
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats()

In [3]:
from pathlib import Path
import json
import uuid
import random
import numpy as np
import pandas as pd
import torch

from transformers import pipeline
from setfit import SetFitModel
from sklearn.metrics import accuracy_score, f1_score, classification_report

RANDOM_SEED = 42
TOP_K = 3
LOW_CONFIDENCE_THRESHOLD = 0.65

PROJECT_DIR = Path("..") if Path("../data").exists() else Path(".")
DATA_SPLITS = PROJECT_DIR / "data" / "splits"
SETFIT_DIR = PROJECT_DIR / "models" / "setfit_intent" / "best_model"
SETFIT_LABEL_MAP_PATHS = [
     PROJECT_DIR / "models" / "setfit_intent" / "label_mappings.json",
     PROJECT_DIR / "models" / "setfit_intent" / "label_mappings",
]

EXPORT_DIR = PROJECT_DIR / "reports" / "rag_export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COL = "instruction"
LABEL_COL = "label"

device =  "cpu"
print("Device:", device)
print("Project dir:", PROJECT_DIR.resolve())

W0510 00:52:10.539540 9204 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Device: cpu
Project dir: D:\conv_nlp_pipeline


## 2. Load test split and SetFit label mapping

In [4]:
test_df = pd.read_csv(DATA_SPLITS / "test.csv")
test_df[TEXT_COL] = test_df[TEXT_COL].astype(str)

if LABEL_COL in test_df.columns:
    test_df[LABEL_COL] = test_df[LABEL_COL].astype(int)

print("Test split shape:", test_df.shape)
display(test_df.head(3))


def load_label_mappings(path: Path) -> tuple[dict[str, int], dict[int, str]]:
    """Load label2id / id2label and normalize JSON key types."""
    with open(path, "r", encoding="utf-8") as f:
        maps = json.load(f)

    label2id = {str(k): int(v) for k, v in maps["label2id"].items()}

    if "id2label" in maps:
        id2label = {int(k): str(v) for k, v in maps["id2label"].items()}
    else:
        id2label = {v: k for k, v in label2id.items()}

    return label2id, id2label


setfit_mapping_path = next((p for p in SETFIT_LABEL_MAP_PATHS if p.exists()), None)

if setfit_mapping_path is None:
    raise FileNotFoundError(
        "SetFit label mapping not found. Expected one of: "
        + ", ".join(str(p) for p in SETFIT_LABEL_MAP_PATHS)
    )

LABEL2ID, ID2LABEL = load_label_mappings(setfit_mapping_path)
NUM_LABELS = len(ID2LABEL)
LABELS = [ID2LABEL[i] for i in range(NUM_LABELS)]

print("SetFit label mapping loaded from:", setfit_mapping_path)
print("Number of intents:", NUM_LABELS)
print("First labels:", LABELS[:5])

Test split shape: (13994, 14)


,instruction,response,intent,label,category,flags,source,mapping_method,word_count,char_count,has_placeholder,placeholder_count,is_external,is_heuristic
0,Does this come with everything I need to conne...,"The unit comes with mounting bracket, cheap sc...",product_information,17,unknown,NaN,amazon_single_qna,heuristic,24,117,False,0,True,True
1,Does the doll come with a bonus dvd with four ...,"no, mine did not",product_information,17,unknown,NaN,amazon_single_qna,heuristic,11,55,False,0,True,True
2,I want to change purchase [ORDER],I understand your desire to modify purchase nu...,change_order,4,order,BL,bitext_support,explicit_intent,6,33,True,1,True,False


SetFit label mapping loaded from: ..\models\setfit_intent\label_mappings.json
Number of intents: 37
First labels: ['add_product', 'availability', 'cancel_order', 'change_account', 'change_order']


## 3. Shared helpers for SetFit prediction and evaluation

In [5]:
def top_k_intents(prob_row: np.ndarray, k: int = TOP_K) -> list[dict]:
    """Return the top-k intent names and probabilities for one row."""
    top_ids = np.argsort(prob_row)[::-1][:k]
    return [
        {
            "intent": ID2LABEL[int(idx)],
            "score": round(float(prob_row[int(idx)]), 4),
        }
        for idx in top_ids
    ]


def build_setfit_output(probs: np.ndarray) -> dict:
    """
    Convert a probability matrix into reusable prediction outputs.
    probs shape: (n_samples, n_labels)
    """
    probs = np.asarray(probs, dtype=float)
    pred_ids = probs.argmax(axis=1).astype(int)
    confidence = probs.max(axis=1).astype(float)
    pred_labels = [ID2LABEL[int(i)] for i in pred_ids]
    top_k = [top_k_intents(row, TOP_K) for row in probs]

    return {
        "probs": probs,
        "pred_ids": pred_ids,
        "pred_labels": pred_labels,
        "confidence": confidence,
        "top_k": top_k,
    }


def evaluate_predictions(y_true: list[int], pred_ids: np.ndarray) -> dict:
    """Evaluate predictions using accuracy, macro F1, weighted F1, and a full report."""
    labels = list(range(NUM_LABELS))
    report = classification_report(
        y_true,
        pred_ids,
        labels=labels,
        target_names=LABELS,
        output_dict=True,
        zero_division=0,
    )

    return {
        "accuracy": float(accuracy_score(y_true, pred_ids)),
        "f1_macro": float(f1_score(y_true, pred_ids, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, pred_ids, average="weighted", zero_division=0)),
        "classification_report": report,
    }


def prediction_record(output: dict, index: int) -> dict:
    """Build the intent block stored in each RAG JSON record."""
    return {
        "predicted": output["pred_labels"][index],
        "confidence": round(float(output["confidence"][index]), 4),
        "top_k": output["top_k"][index],
    }

## 4. Load SetFit and run intent inference

In [6]:
setfit_model = SetFitModel.from_pretrained(str(SETFIT_DIR))
print("SetFit loaded from:", SETFIT_DIR)
print("Model head:", type(setfit_model.model_head))

texts = test_df[TEXT_COL].tolist()

print("Running SetFit inference...")
setfit_probs = np.asarray(setfit_model.predict_proba(texts), dtype=float)

# Safety check: probability columns must match label mapping length.
if setfit_probs.shape[1] != NUM_LABELS:
    raise ValueError(
        f"Mismatch: SetFit returned {setfit_probs.shape[1]} probability columns, "
        f"but label mapping contains {NUM_LABELS} labels."
    )

setfit_output = build_setfit_output(setfit_probs)

print("Done. Probability matrix shape:", setfit_probs.shape)
print(
    "Confidence range:",
    f"{setfit_output['confidence'].min():.3f} – {setfit_output['confidence'].max():.3f}",
)
print("Sample prediction:", prediction_record(setfit_output, 0))

D:\conv_nlp_pipeline\venv\lib\site-packages\torch\nn\modules\module.py:1341: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(
`SentenceTransformer._target_device` has been removed, please use `SentenceTransformer.device` instead.


SetFit loaded from: ..\models\setfit_intent\best_model
Model head: <class 'sklearn.linear_model._logistic.LogisticRegression'>
Running SetFit inference...
Done. Probability matrix shape: (13994, 37)
Confidence range: 0.220 – 0.989
Sample prediction: {'predicted': 'product_information', 'confidence': 0.9874, 'top_k': [{'intent': 'product_information', 'score': 0.9874}, {'intent': 'track_refund', 'score': 0.0004}, {'intent': 'submit_product_idea', 'score': 0.0004}]}


## 5. Internal test evaluation

This section evaluates SetFit on the internal split. The result is kept mainly for documentation. The production choice is based on your external tests, where SetFit generalized better than DistilBERT.

In [7]:
if LABEL_COL in test_df.columns:
    y_true = test_df[LABEL_COL].astype(int).tolist()
    setfit_metrics = evaluate_predictions(y_true, setfit_output["pred_ids"])

    metrics_path = EXPORT_DIR / "setfit_internal_metrics.json"
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(setfit_metrics, f, indent=2, ensure_ascii=False)

    print("SetFit internal metrics:")
    print("accuracy    :", round(setfit_metrics["accuracy"], 4))
    print("f1_macro    :", round(setfit_metrics["f1_macro"], 4))
    print("f1_weighted :", round(setfit_metrics["f1_weighted"], 4))
    print("Saved to:", metrics_path)
else:
    setfit_metrics = None
    print(f"Column '{LABEL_COL}' not found. Skipping internal evaluation.")

SetFit internal metrics:
accuracy    : 0.9302
f1_macro    : 0.9413
f1_weighted : 0.9317
Saved to: ..\reports\rag_export\setfit_internal_metrics.json


## 6. Sentiment analysis with RoBERTa

In [ ]:
print("Loading sentiment model...")
sentiment_pipe = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    top_k=None,
    device=0 if device == "cuda" else -1,
)


def get_sentiment_batch(texts: list[str], batch_size: int = 128) -> list[dict]:
    """Return one sentiment label and score per text."""
    results = []

    for i in range(0, len(texts), batch_size):
        batch = [str(t)[:512] for t in texts[i : i + batch_size]]
        outputs = sentiment_pipe(batch)

        for out in outputs:
            top = max(out, key=lambda x: x["score"])
            results.append({
                "label": top["label"].lower(),
                "score": round(float(top["score"]), 4),
            })

        if i % 5000 == 0:
            print(f"  Sentiment processed: {i}/{len(texts)}")

    return results


print("Running sentiment inference...")
sentiment_results = get_sentiment_batch(texts)
print("Done. Sample:", sentiment_results[:3])

Loading sentiment model...


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Running sentiment inference...


## 7. Churn risk heuristic

In [ ]:
CHURN_RISK_MAP = {
    # High risk — unresolved issue may push the customer to leave
    "damaged_delivery": "high",
    "missing_item": "high",
    "wrong_item": "high",
    "cancel_order": "high",
    "contact_human_agent": "high",
    "delete_account": "high",
    "technical_issue": "high",

    # Medium risk — friction exists but is recoverable
    "payment_issue": "medium",
    "request_refund": "medium",
    "return_product": "medium",
    "exchange_product": "medium",
    "change_order": "medium",
    "recover_password": "medium",
    "check_refund_policy": "medium",

    # Low risk — neutral, informational, or purchase-oriented
    "track_order": "low",
    "track_delivery": "low",
    "delivery_time": "low",
    "shipping_costs": "low",
    "product_information": "low",
    "availability": "low",
    "add_product": "low",
    "remove_product": "low",
    "pay": "low",
    "check_payment_methods": "low",
    "create_account": "low",
    "change_account": "low",
    "request_invoice": "low",
    "order_history": "low",
    "sales_period": "low",
    "store_location": "low",
    "store_opening_hours": "low",
    "submit_feedback": "low",
    "submit_product_idea": "low",
    "request_right_to_rectification": "low",
    "track_refund": "low",
    "use_app": "low",
}


def compute_churn_risk(intent: str, confidence: float, sentiment_label: str | None = None) -> str:
    """
    Rule-based churn risk.
    - Starts from intent risk.
    - Low confidence can bump low -> medium.
    - Negative sentiment can bump low -> medium, and medium -> high.
    """
    risk = CHURN_RISK_MAP.get(intent, "low")

    if confidence < LOW_CONFIDENCE_THRESHOLD and risk == "low":
        risk = "medium"

    if sentiment_label == "negative":
        if risk == "low":
            risk = "medium"
        elif risk == "medium":
            risk = "high"

    return risk

## 8. Label taxonomy — intent to RAG category

Edit these categories and keywords according to the document store used by the RAG system.

In [ ]:
LABEL_TAXONOMY = [
    {"intent": "add_product", "rag_category": "cart", "retrieval_keywords": ["add product", "add item to cart", "shopping cart"]},
    {"intent": "availability", "rag_category": "product_catalog", "retrieval_keywords": ["product availability", "in stock", "available product"]},
    {"intent": "cancel_order", "rag_category": "order_management", "retrieval_keywords": ["cancel order", "order cancellation", "stop order"]},
    {"intent": "change_account", "rag_category": "account", "retrieval_keywords": ["change account", "update account", "edit profile"]},
    {"intent": "change_order", "rag_category": "order_management", "retrieval_keywords": ["modify order", "change order", "update order"]},
    {"intent": "check_payment_methods", "rag_category": "payment", "retrieval_keywords": ["payment methods", "accepted payment", "pay by card"]},
    {"intent": "check_refund_policy", "rag_category": "returns_refunds", "retrieval_keywords": ["refund policy", "refund conditions", "money back"]},
    {"intent": "contact_human_agent", "rag_category": "support", "retrieval_keywords": ["human agent", "customer support", "contact support"]},
    {"intent": "create_account", "rag_category": "account", "retrieval_keywords": ["create account", "sign up", "new account"]},
    {"intent": "damaged_delivery", "rag_category": "delivery", "retrieval_keywords": ["damaged delivery", "damaged package", "broken item"]},
    {"intent": "delete_account", "rag_category": "account", "retrieval_keywords": ["delete account", "close account", "remove profile"]},
    {"intent": "delivery_time", "rag_category": "delivery", "retrieval_keywords": ["delivery time", "estimated delivery", "shipping duration"]},
    {"intent": "exchange_product", "rag_category": "returns_refunds", "retrieval_keywords": ["exchange product", "replace item", "product exchange"]},
    {"intent": "missing_item", "rag_category": "delivery", "retrieval_keywords": ["missing item", "incomplete order", "item not received"]},
    {"intent": "order_history", "rag_category": "order_management", "retrieval_keywords": ["order history", "past orders", "previous purchases"]},
    {"intent": "pay", "rag_category": "payment", "retrieval_keywords": ["pay order", "checkout payment", "complete payment"]},
    {"intent": "payment_issue", "rag_category": "payment", "retrieval_keywords": ["payment issue", "payment failed", "card declined"]},
    {"intent": "product_information", "rag_category": "product_catalog", "retrieval_keywords": ["product information", "product details", "item description"]},
    {"intent": "product_issue", "rag_category": "product_support", "retrieval_keywords": ["product issue", "defective product", "problem with product"]},
    {"intent": "recover_password", "rag_category": "account", "retrieval_keywords": ["recover password", "reset password", "forgot password"]},
    {"intent": "remove_product", "rag_category": "cart", "retrieval_keywords": ["remove product", "remove item", "delete from cart"]},
    {"intent": "request_invoice", "rag_category": "billing", "retrieval_keywords": ["request invoice", "billing document", "receipt"]},
    {"intent": "request_refund", "rag_category": "returns_refunds", "retrieval_keywords": ["request refund", "get refund", "refund request"]},
    {"intent": "request_right_to_rectification", "rag_category": "privacy", "retrieval_keywords": ["rectification right", "correct personal data", "GDPR correction"]},
    {"intent": "return_product", "rag_category": "returns_refunds", "retrieval_keywords": ["return product", "return item", "send item back"]},
    {"intent": "sales_period", "rag_category": "sales", "retrieval_keywords": ["sales period", "discount period", "promotion dates"]},
    {"intent": "shipping_costs", "rag_category": "delivery", "retrieval_keywords": ["shipping costs", "delivery fee", "shipping price"]},
    {"intent": "store_location", "rag_category": "store_info", "retrieval_keywords": ["store location", "nearest store", "shop address"]},
    {"intent": "store_opening_hours", "rag_category": "store_info", "retrieval_keywords": ["opening hours", "store hours", "business hours"]},
    {"intent": "submit_feedback", "rag_category": "feedback", "retrieval_keywords": ["submit feedback", "customer feedback", "review service"]},
    {"intent": "submit_product_idea", "rag_category": "feedback", "retrieval_keywords": ["product idea", "suggest product", "new product suggestion"]},
    {"intent": "technical_issue", "rag_category": "technical_support", "retrieval_keywords": ["technical issue", "app problem", "website error"]},
    {"intent": "track_delivery", "rag_category": "delivery", "retrieval_keywords": ["track delivery", "shipment tracking", "package status"]},
    {"intent": "track_order", "rag_category": "order_management", "retrieval_keywords": ["track order", "order status", "where is my order"]},
    {"intent": "track_refund", "rag_category": "returns_refunds", "retrieval_keywords": ["track refund", "refund status", "refund progress"]},
    {"intent": "use_app", "rag_category": "app_help", "retrieval_keywords": ["use app", "app guide", "how to use application"]},
    {"intent": "wrong_item", "rag_category": "delivery", "retrieval_keywords": ["wrong item", "incorrect product", "wrong product received"]},
]

taxonomy_path = EXPORT_DIR / "label_taxonomy.json"
with open(taxonomy_path, "w", encoding="utf-8") as f:
    json.dump(LABEL_TAXONOMY, f, indent=2, ensure_ascii=False)

print("Label taxonomy saved to:", taxonomy_path)
print("Taxonomy rows:", len(LABEL_TAXONOMY))

## 9. Assemble and export final RAG JSON

In [ ]:
records = []

for i, row in test_df.reset_index(drop=True).iterrows():
    intent_block = prediction_record(setfit_output, i)
    pred_intent = intent_block["predicted"]
    confidence = intent_block["confidence"]
    sent = sentiment_results[i]
    churn_level = compute_churn_risk(pred_intent, confidence, sent["label"])

    record = {
        "record_id": str(uuid.uuid4()),
        "clean_instruction": row[TEXT_COL],

        "intent": {
            **intent_block,
            "primary_model": "SetFit",
            "method": "setfit_sentence_embeddings",
        },

        "sentiment": {
            "label": sent["label"],
            "score": sent["score"],
            "method": "roberta-zero-shot",
        },

        "churn_risk": {
            "level": churn_level,
            "method": "intent-confidence-sentiment-heuristic",
        },
    }

    if LABEL_COL in row:
        record["ground_truth"] = {
            "label_id": int(row[LABEL_COL]),
            "label_name": ID2LABEL.get(int(row[LABEL_COL]), None),
        }

    records.append(record)

    if i % 5000 == 0:
        print(f"  {i}/{len(test_df)} records assembled...")

rag_output_path = EXPORT_DIR / "rag_output.json"
with open(rag_output_path, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

print("Total records:", len(records))
print("RAG output saved to:", rag_output_path)

## 10. Sanity checks

In [ ]:
random.seed(RANDOM_SEED)
samples = random.sample(records, min(3, len(records)))

for sample in samples:
    print(json.dumps(sample, indent=2, ensure_ascii=False))
    print("-" * 80)

In [ ]:
intents_dist = pd.Series([r["intent"]["predicted"] for r in records]).value_counts()
sentiment_dist = pd.Series([r["sentiment"]["label"] for r in records]).value_counts()
churn_dist = pd.Series([r["churn_risk"]["level"] for r in records]).value_counts()
confidence_stats = pd.Series([r["intent"]["confidence"] for r in records]).describe()

print("Top predicted intents:")
display(intents_dist.head(10).to_frame("count"))

print("Sentiment distribution:")
display(sentiment_dist.to_frame("count"))

print("Churn risk distribution:")
display(churn_dist.to_frame("count"))

print("Confidence statistics:")
display(confidence_stats.to_frame("confidence"))

## 11. Export summary

In [ ]:
files = [
    (EXPORT_DIR / "rag_output.json", "Main RAG JSON — one record per customer message"),
    (EXPORT_DIR / "setfit_internal_metrics.json", "SetFit internal metrics on test split"),
    (EXPORT_DIR / "label_taxonomy.json", "Intent to RAG category and retrieval keywords"),
]

print("Files exported for RAG:")
print(f"{'File':<55} {'Size':>10}  Description")
print("-" * 110)

for path, desc in files:
    size = f"{path.stat().st_size / 1024:.1f} KB" if path.exists() else "MISSING"
    print(f"{str(path):<55} {size:>10}  {desc}")

## Notes for the report

Use this explanation in the methodology section:

> DistilBERT achieved stronger performance on the internal test split, but SetFit was selected as the production intent classifier because it showed better generalization on external datasets. Since the final RAG system must handle heterogeneous real user queries, semantic robustness and cross-dataset generalization were prioritized over internal split performance.

This notebook therefore removes the automatic model-selection logic and uses SetFit directly as the primary intent model.